# 00 – Diagnóstico dos dados (canvas de catalogação)

Levantamento dos números para o canvas de catalogação de dados.
Fonte: Série Histórica de Preços de Combustíveis da ANP (`data/raw/`) e a saída
já processada do `src/load.py` (`data/processed/precos_al.parquet`).

**Não faz gráfico e não modela nada.** Três blocos — FORMATO, FREQUÊNCIA E
LATÊNCIA, QUALIDADE — mais um resumo em texto no fim para colar nos post-its.

Rodar de cima para baixo. Os números vão sendo guardados em `R` e a última
célula imprime tudo junto.

In [1]:
# --- setup -------------------------------------------------------------------
import sys, zipfile, io
from pathlib import Path

import numpy as np
import pandas as pd

# raiz do projeto, funcione o notebook rodando de notebooks/ ou da raiz
RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
import load as L            # reusa ler_csv/ler_zip, normalizar_nome, RENOMEAR, limpar, validar

RAW = RAIZ / "data" / "raw"
PROCESSED = RAIZ / "data" / "processed"
PARQUET = PROCESSED / "precos_al.parquet"

HOJE = pd.Timestamp.today().normalize()   # data em que o notebook roda (dia da aula)

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

R = {}   # coletor de números para o resumo final

arquivos_raw = sorted(p for p in RAW.iterdir() if p.suffix in (".csv", ".zip"))
al = pd.read_parquet(PARQUET)

print("raiz do projeto :", RAIZ)
print("arquivos em data/raw :", len(arquivos_raw))
print("hoje (referência da latência) :", HOJE.date())
print("parquet :", PARQUET.name, "->", f"{len(al):,}".replace(",", "."), "linhas x", al.shape[1], "colunas")

raiz do projeto : C:\Users\jamil\OneDrive\Área de Trabalho\Academic\curso-ciencia-dados-oxetech\projeto-gasolina-al
arquivos em data/raw : 16
hoje (referência da latência) : 2026-09-03
parquet : precos_al.parquet -> 30.091 linhas x 18 colunas


## 1. FORMATO

In [2]:
# === BLOCO 1: FORMATO ======================================================
# - colunas de um arquivo antigo (pré-2022) vs. um recente, lado a lado
# - tipo de cada coluna e tamanho dos arquivos em MB
# - total de linhas no arquivo nacional vs. total após filtrar AL + gasolina

ANTIGO  = RAW / "ca-2019-01.csv"          # .csv solto, latin-1  (formato pré-2022)
RECENTE = RAW / "ca-2026-01.zip"          # .zip, utf-8          (formato atual)


def colunas_cruas(caminho: Path) -> list[str]:
    """Cabeçalho exatamente como a ANP entrega, sem normalizar."""
    if caminho.suffix == ".zip":
        with zipfile.ZipFile(caminho) as zf:
            nome = next(n for n in zf.namelist() if n.lower().endswith(".csv"))
            for enc in ("utf-8-sig", "latin-1"):
                try:
                    with zf.open(nome) as f:
                        return pd.read_csv(f, sep=";", encoding=enc, nrows=0).columns.tolist()
                except UnicodeDecodeError:
                    continue
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(caminho, sep=";", encoding=enc, nrows=0).columns.tolist()
        except UnicodeDecodeError:
            continue
    raise ValueError(caminho.name)


col_antigo  = colunas_cruas(ANTIGO)
col_recente = colunas_cruas(RECENTE)

print("=" * 78)
print(f"1a. COLUNAS CRUAS  —  {ANTIGO.name}  vs  {RECENTE.name}")
print("=" * 78)
larg = max([len(c) for c in col_antigo + col_recente] + [27]) + 2
print(f"{'#':>2}  {'arquivo antigo (2019-01)':<{larg}}{'arquivo recente (2026-01)':<{larg}}igual?")
print("-" * 78)
divergencias = []
for i in range(max(len(col_antigo), len(col_recente))):
    a = col_antigo[i]  if i < len(col_antigo)  else "—"
    b = col_recente[i] if i < len(col_recente) else "—"
    ok = "sim" if a == b else "NÃO"
    if a != b:
        divergencias.append((a, b))
    print(f"{i:>2}  {a:<{larg}}{b:<{larg}}{ok}")
print("-" * 78)
if divergencias:
    print(f"{len(divergencias)} coluna(s) com nome divergente entre os dois arquivos:")
    for a, b in divergencias:
        print(f"   {a!r:40} !=  {b!r}")
else:
    print("Nenhuma divergência: os dois arquivos usam exatamente o mesmo cabeçalho.")

# a divergência real do projeto é entre o rótulo da ANP e o nome interno
print()
print("=" * 78)
print("1b. DE-PARA  rótulo da ANP  ->  nome normalizado  ->  nome interno do load.py")
print("=" * 78)
print(f"{'rótulo cru da ANP':<26}{'normalizar_nome()':<26}{'nome final (RENOMEAR)':<24}")
print("-" * 78)
remapeadas = 0    # o RENOMEAR troca o nome (além de virar snake_case)
sem_mapa = 0      # nem no RENOMEAR: fica só com a versão normalizada
for c in col_recente:
    n = L.normalizar_nome(c)
    final = L.RENOMEAR.get(n, n)
    if n not in L.RENOMEAR:
        sem_mapa += 1
        marca = "  (sem mapeamento)"
    elif final != n:
        remapeadas += 1
        marca = "  <- renomeada"
    else:
        marca = ""
    print(f"{c:<26}{n:<26}{final:<24}{marca}")
print("-" * 78)
print(f"As {len(col_recente)} colunas viram snake_case (tira acento, hífen e espaço).")
print(f"   {remapeadas} são renomeadas de fato pelo RENOMEAR "
      f"(ex.: estado_sigla->uf, valor_de_venda->valor_venda).")
print(f"   {sem_mapa} não têm entrada no RENOMEAR e seguem só com o nome normalizado "
      f"(nome_da_rua, numero_rua, complemento, cep).")

# --- tipos ----------------------------------------------------------------
print()
print("=" * 78)
print("1c. TIPO DE CADA COLUNA")
print("=" * 78)
print("No RAW da ANP toda coluna vem como texto (o load.py lê com dtype=str).")
print("Tipos abaixo são os do parquet já processado (data/processed/precos_al.parquet):")
print()
print(al.dtypes.to_string())

# --- tamanho dos arquivos ----------------------------------------------------
print()
print("=" * 78)
print("1d. TAMANHO DOS ARQUIVOS EM DISCO (MB)")
print("=" * 78)
tam = []
for p in arquivos_raw:
    tam.append((p.name, p.stat().st_size / 1e6))
for nome, mb in tam:
    print(f"   {nome:<42}{mb:8.1f} MB")
mb_parquet = PARQUET.stat().st_size / 1e6
print("-" * 60)
print(f"   {'TOTAL data/raw/':<42}{sum(m for _, m in tam):8.1f} MB")
print(f"   {'data/processed/' + PARQUET.name:<42}{mb_parquet:8.2f} MB")

# --- linhas: nacional vs. AL + gasolina -------------------------------------
print()
print("=" * 78)
print("1e. LINHAS: ARQUIVO NACIONAL  vs.  APÓS FILTRAR AL + GASOLINA")
print("=" * 78)


def contar(caminho: Path) -> tuple[int, int]:
    bruto = L.carregar_arquivo(caminho)
    recorte = bruto[
        (bruto["uf"].str.strip().str.upper() == "AL")
        & (bruto["produto"].str.upper().str.startswith("GASOLINA"))
    ]
    return len(bruto), len(recorte)


nac_ant, al_ant = contar(ANTIGO)
nac_rec, al_rec = contar(RECENTE)
for nome, nac, alf in [(ANTIGO.name, nac_ant, al_ant), (RECENTE.name, nac_rec, al_rec)]:
    print(f"   {nome:<24} nacional: {nac:>9,}   AL+gasolina: {alf:>7,}   "
          f"({100*alf/nac:.2f}%)".replace(",", "."))
print()
print(f"   parquet final (todos os {len(arquivos_raw)} arquivos concatenados): "
      f"{len(al):,} linhas".replace(",", "."))

R["formato"] = dict(
    antigo=ANTIGO.name, recente=RECENTE.name,
    n_colunas=len(col_recente), cabecalhos_iguais=not divergencias,
    divergencias=divergencias, remapeadas=remapeadas, sem_mapa=sem_mapa,
    tam_raw_total_mb=sum(m for _, m in tam), tam_parquet_mb=mb_parquet,
    tam_por_arquivo=tam,
    nac_antigo=nac_ant, al_antigo=al_ant,
    nac_recente=nac_rec, al_recente=al_rec,
    linhas_parquet=len(al),
)

1a. COLUNAS CRUAS  —  ca-2019-01.csv  vs  ca-2026-01.zip
 #  arquivo antigo (2019-01)     arquivo recente (2026-01)    igual?
------------------------------------------------------------------------------
 0  Regiao - Sigla               Regiao - Sigla               sim
 1  Estado - Sigla               Estado - Sigla               sim
 2  Municipio                    Municipio                    sim
 3  Revenda                      Revenda                      sim
 4  CNPJ da Revenda              CNPJ da Revenda              sim
 5  Nome da Rua                  Nome da Rua                  sim
 6  Numero Rua                   Numero Rua                   sim
 7  Complemento                  Complemento                  sim
 8  Bairro                       Bairro                       sim
 9  Cep                          Cep                          sim
10  Produto                      Produto                      sim
11  Data da Coleta               Data da Coleta               sim
12 

   ca-2019-01.csv           nacional:   496.730   AL+gasolina:   1.481   (0.30%)
   ca-2026-01.zip           nacional:   422.418   AL+gasolina:   2.529   (0.60%)

   parquet final (todos os 16 arquivos concatenados): 30.091 linhas


## 2. FREQUÊNCIA E LATÊNCIA

In [3]:
# === BLOCO 2: FREQUÊNCIA E LATÊNCIA ========================================
# - data da coleta mais recente na base
# - diferença em dias entre essa data e hoje  (latência real)
# - intervalo típico entre coletas consecutivas
# - quantas semanas do calendário faltam na série

datas   = al["data_coleta"]
semanas = al["semana"].dropna().sort_values()

coleta_recente = datas.max()
coleta_antiga  = datas.min()
latencia_dias  = (HOJE - coleta_recente).days

print("=" * 78)
print("2a. DATA DA COLETA MAIS RECENTE / LATÊNCIA")
print("=" * 78)
print(f"   primeira coleta na base : {coleta_antiga.date()}")
print(f"   última   coleta na base : {coleta_recente.date()}")
print(f"   hoje (dia do download)  : {HOJE.date()}")
print(f"   LATÊNCIA (1 medição)    : {latencia_dias} dias "
      f"(~{latencia_dias/7:.1f} semanas) — dado mais novo vs. hoje.")
print(f"   medido contra o dia do download, não da publicação => é um TETO: "
      f"latência do registro mais recente <= {latencia_dias} dias.")
print(f"   não verificado se é estável (premissa CLAUDE.md: ~1 a 2 semanas).")

# --- intervalo entre coletas ----------------------------------------------
print()
print("=" * 78)
print("2b. INTERVALO ENTRE COLETAS CONSECUTIVAS")
print("=" * 78)

dias_distintos = pd.Series(sorted(datas.dt.normalize().unique()))
dif_dias = dias_distintos.diff().dropna().dt.days

sem_distintas = pd.Series(sorted(semanas.unique()))
dif_sem = sem_distintas.diff().dropna().dt.days

print("A ANP faz coleta de campo quase todo dia útil, mas consolida por semana.")
print()
print(f"   entre DIAS de coleta distintos : mediana {dif_dias.median():.0f} dia(s), "
      f"moda {dif_dias.mode().tolist()}, máx {int(dif_dias.max())} dias")
print(f"   entre SEMANAS de referência    : mediana {dif_sem.median():.0f} dias, "
      f"moda {dif_sem.mode().tolist()}, máx {int(dif_sem.max())} dias")
print()
print("   => o passo natural da série é SEMANAL (7 dias).")

# --- semanas faltando ----------------------------------------------------
print()
print("=" * 78)
print("2c. SEMANAS DO CALENDÁRIO FALTANDO NA SÉRIE")
print("=" * 78)
# load.py ancora a semana na segunda-feira (period W-SUN -> start_time)
calendario = pd.date_range(sem_distintas.min(), sem_distintas.max(), freq="W-MON")
presentes = set(sem_distintas)
faltando = [d for d in calendario if d not in presentes]

print(f"   intervalo coberto : {sem_distintas.min().date()} a {sem_distintas.max().date()}")
print(f"   semanas esperadas : {len(calendario)}")
print(f"   semanas presentes : {len(sem_distintas)}")
print(f"   SEMANAS FALTANDO  : {len(faltando)}")
if faltando:
    print("   datas (segunda-feira da semana ausente):")
    for d in faltando:
        print(f"      {d.date()}")

R["frequencia"] = dict(
    coleta_recente=coleta_recente, coleta_antiga=coleta_antiga,
    hoje=HOJE, latencia_dias=latencia_dias,
    intervalo_dias_mediana=float(dif_dias.median()),
    intervalo_semana_mediana=float(dif_sem.median()),
    intervalo_semana_max=int(dif_sem.max()),
    semanas_esperadas=len(calendario), semanas_presentes=len(sem_distintas),
    semanas_faltando=len(faltando), datas_faltando=[d.date() for d in faltando],
)

2a. DATA DA COLETA MAIS RECENTE / LATÊNCIA
   primeira coleta na base : 2019-01-02
   última   coleta na base : 2026-08-26
   hoje (dia do download)  : 2026-09-03
   LATÊNCIA (1 medição)    : 8 dias (~1.1 semanas) — dado mais novo vs. hoje.
   medido contra o dia do download, não da publicação => é um TETO: latência do registro mais recente <= 8 dias.
   não verificado se é estável (premissa CLAUDE.md: ~1 a 2 semanas).

2b. INTERVALO ENTRE COLETAS CONSECUTIVAS
A ANP faz coleta de campo quase todo dia útil, mas consolida por semana.

   entre DIAS de coleta distintos : mediana 1 dia(s), moda [1], máx 63 dias
   entre SEMANAS de referência    : mediana 7 dias, moda [7], máx 63 dias

   => o passo natural da série é SEMANAL (7 dias).

2c. SEMANAS DO CALENDÁRIO FALTANDO NA SÉRIE
   intervalo coberto : 2018-12-31 a 2026-08-24
   semanas esperadas : 400
   semanas presentes : 387
   SEMANAS FALTANDO  : 13
   datas (segunda-feira da semana ausente):
      2020-08-24
      2020-08-31
      202

## 3. QUALIDADE

In [4]:
# === BLOCO 3: QUALIDADE ====================================================
# - % de nulos por coluna na base de AL
# - em que data 'valor_compra' para de existir
# - contagem de linhas rejeitadas por motivo (regras do load.py)
# - mín / máx / média / desvio de 'valor_venda' por ano
# - quantos valores distintos 'bandeira' assume, e quais

print("=" * 78)
print("3a. PERCENTUAL DE NULOS POR COLUNA — base de AL (parquet)")
print("=" * 78)
nulos = (al.isna().mean() * 100).round(2)
for col, pct in nulos.items():
    marca = "   <-- ausência estrutural" if col == "valor_compra" else ""
    print(f"   {col:<18}{pct:6.2f} %{marca}")

# --- valor_compra: quando some -------------------------------------------
print()
print("=" * 78)
print("3b. 'valor_compra' — CONFIRMAÇÃO DA DATA EM QUE PARA DE EXISTIR")
print("=" * 78)
com_compra = al[al["valor_compra"].notna()]
por_ano_vc = al.groupby(al["data_coleta"].dt.year)["valor_compra"].apply(lambda s: s.notna().sum())
print("   linhas com valor_compra preenchido, por ano:")
for ano, n in por_ano_vc.items():
    print(f"      {ano}: {n:>5}")
print()
vc_ultima = com_compra["data_coleta"].max()
vc_prox = al[al["data_coleta"] > vc_ultima]["data_coleta"].min()
print(f"   ÚLTIMA coleta com valor_compra preenchido : {vc_ultima.date()}")
print(f"   coleta seguinte na base (já 100% vazia)   : {vc_prox.date()}")
print(f"   CLAUDE.md diz 'some a partir de 2020'; os dados mostram o corte exato "
      f"em {vc_ultima.date()}.")
print("   => de 2021 em diante é ausência estrutural (a ANP parou de divulgar), não dado faltante.")

# --- rejeitados por motivo ----------------------------------------------
print()
print("=" * 78)
print("3c. LINHAS REJEITADAS POR MOTIVO (regras de validar() do load.py)")
print("=" * 78)
rej_path = PROCESSED / "rejeitados.csv"
if rej_path.exists():
    rej = pd.read_csv(rej_path)
    contagem = rej["motivo_rejeicao"].value_counts()
    total_rej = len(rej)
    print(f"   arquivo: {rej_path.name}  ({total_rej} linhas)")
    for motivo, n in contagem.items():
        print(f"      {motivo:<32}{n:>7}")
else:
    # o load.py só grava rejeitados.csv quando há rejeições; refaz a conta aqui
    partes = []
    for p in arquivos_raw:
        b = L.carregar_arquivo(p)
        partes.append(b[(b["uf"].str.strip().str.upper() == "AL")
                        & (b["produto"].str.upper().str.startswith("GASOLINA"))])
    full = L.limpar(pd.concat(partes, ignore_index=True))
    validos, rej = L.validar(full)
    total_rej = len(rej)
    print(f"   {rej_path.name} não existe — o load.py não gravou porque não houve rejeição.")
    print(f"   Refazendo validar() sobre AL+gasolina ({len(full):,} linhas):".replace(",", "."))
    motivos = ["data invalida", "preco ausente", "preco nao positivo",
               "preco fora da faixa plausivel", "cnpj malformado"]
    if total_rej:
        for motivo, n in rej["motivo_rejeicao"].value_counts().items():
            print(f"      {motivo:<32}{n:>7}")
    else:
        for motivo in motivos:
            print(f"      {motivo:<32}{0:>7}")
        print("   => nenhuma linha rejeitada. As regras atuais (faixa 0,80–12,00, "
              "CNPJ 14 díg., data dd/mm/aaaa) não pegam nada em AL.")

# --- valor_venda por ano ----------------------------------------------
print()
print("=" * 78)
print("3d. valor_venda POR ANO  (R$/litro) — mín / máx / média / desvio")
print("=" * 78)
stats = al.groupby(al["data_coleta"].dt.year)["valor_venda"].agg(
    ["min", "max", "mean", "std", "count"]
)
print(stats.round(3).to_string())

# --- bandeira ---------------------------------------------------------
print()
print("=" * 78)
print("3e. BANDEIRA — quantos valores distintos e quais")
print("=" * 78)
band = al["bandeira"].value_counts(dropna=False)
print(f"   valores distintos: {al['bandeira'].nunique(dropna=True)}")
print()
for b, n in band.items():
    print(f"      {str(b):<32}{n:>7}")
print()
print("   Obs.: há redundância que precisa de de-para (armadilha nº5) —")
print("   VIBRA / VIBRA ENERGIA / PETROBRAS DISTRIBUIDORA S.A. são a mesma linhagem;")
print("   ALE / ALESAT idem. 'BRANCA' = posto sem bandeira (bandeira branca).")

R["qualidade"] = dict(
    nulos=nulos.to_dict(),
    vc_ultima_data=vc_ultima.date(),
    vc_por_ano=por_ano_vc.to_dict(),
    rejeitados_total=int(total_rej),
    vv_por_ano=stats.round(3).to_dict("index"),
    bandeira_n=int(al["bandeira"].nunique(dropna=True)),
    bandeira_valores=band.to_dict(),
)

3a. PERCENTUAL DE NULOS POR COLUNA — base de AL (parquet)
   regiao              0.00 %
   uf                  0.00 %
   municipio           0.00 %
   revenda             0.00 %
   cnpj                0.00 %
   nome_da_rua         0.00 %
   numero_rua          0.00 %
   complemento        74.45 %
   bairro              0.00 %
   cep                 0.00 %
   produto             0.00 %
   data_coleta         0.00 %
   valor_venda         0.00 %
   valor_compra       92.81 %   <-- ausência estrutural
   unidade             0.00 %
   bandeira            0.00 %
   arquivo_origem      0.00 %
   semana              0.00 %

3b. 'valor_compra' — CONFIRMAÇÃO DA DATA EM QUE PARA DE EXISTIR
   linhas com valor_compra preenchido, por ano:
      2019:  1408
      2020:   757
      2021:     0
      2022:     0
      2023:     0
      2024:     0
      2025:     0
      2026:     0

   ÚLTIMA coleta com valor_compra preenchido : 2020-08-17
   coleta seguinte na base (já 100% vazia)   : 2020-10-19
  

   rejeitados.csv não existe — o load.py não gravou porque não houve rejeição.
   Refazendo validar() sobre AL+gasolina (30.091 linhas):
      data invalida                         0
      preco ausente                         0
      preco nao positivo                    0
      preco fora da faixa plausivel         0
      cnpj malformado                       0
   => nenhuma linha rejeitada. As regras atuais (faixa 0,80–12,00, CNPJ 14 díg., data dd/mm/aaaa) não pegam nada em AL.

3d. valor_venda POR ANO  (R$/litro) — mín / máx / média / desvio
              min   max  mean   std  count
data_coleta                               
2019        3.870 5.310 4.579 0.238   2998
2020        3.499 5.210 4.474 0.350   2463
2021        4.450 7.557 6.004 0.585   3654
2022        4.490 8.590 6.496 0.929   4052
2023        4.090 7.070 5.799 0.417   4770
2024        5.280 7.230 6.158 0.408   4952
2025        5.190 7.480 6.409 0.343   4276
2026        5.440 8.040 6.775 0.448   2926

3e. BANDEIRA — q

## 4. Resumo para os post-its

In [5]:
# === RESUMO EM TEXTO — copiar para os post-its =============================
f = R["formato"]; q = R["frequencia"]; z = R["qualidade"]
L_ = "-" * 70

print(L_)
print("CANVAS DE CATALOGAÇÃO — GASOLINA/AL — ANP")
print(f"gerado em {HOJE.date()}  |  fonte: Série Histórica de Preços de Combustíveis (ANP)")
print(L_)

print("""
[ FORMATO ]""")
print(f"- Arquivos brutos: {len(arquivos_raw)} (2019–2026). Antigo comparado: {f['antigo']}"
      f" | recente: {f['recente']}.")
print(f"- Colunas: {f['n_colunas']} em todos. Cabeçalho cru do arquivo antigo e do recente"
      f" {'é IDÊNTICO' if f['cabecalhos_iguais'] else 'DIVERGE'} "
      f"(nenhuma coluna com nome diferente entre eles).")
print(f"- Divergência real = rótulo da ANP vs. nome interno: todas as {f['n_colunas']} viram"
      f" snake_case; {f['remapeadas']} são renomeadas de fato pelo RENOMEAR do load.py"
      f" (ex.: 'Estado - Sigla'->'uf', 'CNPJ da Revenda'->'cnpj'); {f['sem_mapa']} sem mapeamento.")
n_txt = sum(str(t) not in ("datetime64[us]", "float64") for t in al.dtypes)
print(f"- Tipos (parquet, {al.shape[1]} colunas): {n_txt} como texto, data_coleta e semana"
      f" como datetime64, valor_venda e valor_compra como float64."
      f" No RAW da ANP tudo vem como texto.")
print(f"- Tamanho: data/raw/ = {f['tam_raw_total_mb']:.0f} MB no total"
      f" (csv 2019–2021 ~40–90 MB cada; zip 2022–2026 ~7–17 MB);"
      f" parquet processado = {f['tam_parquet_mb']:.2f} MB.")
print(f"- Linhas nacional vs. AL+gasolina:")
print("    {}: {} -> {}".format(f['antigo'],
      f"{f['nac_antigo']:,}".replace(',', '.'), f"{f['al_antigo']:,}".replace(',', '.')))
print("    {}: {} -> {}".format(f['recente'],
      f"{f['nac_recente']:,}".replace(',', '.'), f"{f['al_recente']:,}".replace(',', '.')))
print("    parquet final (AL + gasolina, os {} arquivos): {} linhas".format(
      len(arquivos_raw), f"{f['linhas_parquet']:,}".replace(',', '.')))

print("""
[ FREQUÊNCIA E LATÊNCIA ]""")
print(f"- Coleta mais recente na base: {q['coleta_recente'].date()}  "
      f"(série começa em {q['coleta_antiga'].date()}).")
print(f"- LATÊNCIA (1 medição): dado mais novo com {q['latencia_dias']} dias em "
      f"{HOJE.date()}; medido vs. dia do download => teto <= {q['latencia_dias']} dias. "
      f"Estabilidade não verificada (premissa CLAUDE.md: ~1 a 2 sem).")
print(f"- Intervalo entre coletas: passo semanal (mediana {q['intervalo_semana_mediana']:.0f} dias "
      f"entre semanas; {q['intervalo_dias_mediana']:.0f} dia entre dias de campo).")
print(f"- Semanas do calendário faltando: {q['semanas_faltando']} de {q['semanas_esperadas']} "
      f"esperadas ({q['semanas_presentes']} presentes). Maior buraco: {q['intervalo_semana_max']} dias.")

print("""
[ QUALIDADE ]""")
print(f"- Nulos por coluna (base AL): complemento {z['nulos']['complemento']:.1f}%, "
      f"valor_compra {z['nulos']['valor_compra']:.1f}%; todas as demais 0%.")
print(f"- valor_compra deixa de existir em {z['vc_ultima_data']} "
      f"(última coleta com o campo preenchido). Preenchido só em 2019 "
      f"({z['vc_por_ano'].get(2019,0)}) e 2020 ({z['vc_por_ano'].get(2020,0)}); "
      f"0 de 2021 em diante. Ausência estrutural, não imputar.")
print(f"- Linhas rejeitadas pela validação do load.py: {z['rejeitados_total']} "
      f"(não gerou rejeitados.csv).")
print(f"- valor_venda por ano (R$/L)  min | max | média | desvio:")
for ano, d in z['vv_por_ano'].items():
    print(f"    {ano}: {d['min']:.2f} | {d['max']:.2f} | {d['mean']:.2f} | {d['std']:.2f}"
          f"   (n={int(d['count'])})")
print(f"- bandeira: {z['bandeira_n']} valores distintos ->")
print(f"    {', '.join(f'{k} ({v})' for k, v in z['bandeira_valores'].items())}")
print(f"    precisa de de-para: VIBRA/VIBRA ENERGIA/PETROBRAS DISTRIBUIDORA S.A. e ALE/ALESAT.")
print(L_)

----------------------------------------------------------------------
CANVAS DE CATALOGAÇÃO — GASOLINA/AL — ANP
gerado em 2026-09-03  |  fonte: Série Histórica de Preços de Combustíveis (ANP)
----------------------------------------------------------------------

[ FORMATO ]
- Arquivos brutos: 16 (2019–2026). Antigo comparado: ca-2019-01.csv | recente: ca-2026-01.zip.
- Colunas: 16 em todos. Cabeçalho cru do arquivo antigo e do recente é IDÊNTICO (nenhuma coluna com nome diferente entre eles).
- Divergência real = rótulo da ANP vs. nome interno: todas as 16 viram snake_case; 7 são renomeadas de fato pelo RENOMEAR do load.py (ex.: 'Estado - Sigla'->'uf', 'CNPJ da Revenda'->'cnpj'); 4 sem mapeamento.
- Tipos (parquet, 18 colunas): 14 como texto, data_coleta e semana como datetime64, valor_venda e valor_compra como float64. No RAW da ANP tudo vem como texto.
- Tamanho: data/raw/ = 536 MB no total (csv 2019–2021 ~40–90 MB cada; zip 2022–2026 ~7–17 MB); parquet processado = 0.24 MB.
- Linh